## Update Yearly Like playlists wit Musicbee likes

In [13]:
import time

import pandas as pd
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)

from ytmusic_library import YTMusicPlaylists

DATE = time.strftime('%m-%d-%Y')

HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)

uni_module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if uni_module_path not in sys.path: sys.path.append(uni_module_path)
import unify_lib as uni

# MB_LIB = os.path.join(module_path, 'db_assets/musicbee_library.tsv')
# MB_INBOX = os.path.join(module_path, 'db_assets/musicbee_inbox.tsv')

# print('Loading ytmusic and musicbee track databases (takes ~1m)')
# mb_tracks = uni.ingest_musicbee_db_assets(
#     MB_LIB, MB_INBOX, save_tsv=False
# )

# YT_TRACK_DB = os.path.join(PLAYLIST_TSV_DIR, '_tracks_db.tsv')
# yt_tracks = uni.ingest_ytmusic_db_assets( YT_TRACK_DB, save_tsv=False)

ALBUM_TRACK_MATCH_TSV = os.path.join(
    uni_module_path, 'tsvs', 'musicbee_track_matches_for_yt_album_matches.tsv'
)

ARTIST_TRACK_MATCH_TSV = os.path.join(
    uni_module_path, 'tsvs', 'musicbee_track_matches_for_yt_artist_matches.tsv'
)
artist_track_matches = pd.read_csv(ARTIST_TRACK_MATCH_TSV, sep='\t')
album_track_matches = pd.read_csv(ALBUM_TRACK_MATCH_TSV, sep='\t')
artist_track_matches = artist_track_matches.loc[
    artist_track_matches['mb_match_label'] == 'MATCH'
]
album_track_matches = album_track_matches.loc[
    album_track_matches['mb_match_label'] == 'MATCH'
]

# Overwrite artist matches with album matches if same path
mb_path_map = dict(zip(artist_track_matches['mb_Path'], artist_track_matches['yt_videoId']))
mb_path_map.update(dict(zip(album_track_matches['mb_Path'], album_track_matches['yt_videoId']))) 

print(f'{len(mb_path_map)} yt_videoIds have a mb path')


Using header file: ../oauth.json
45601 yt_videoIds have a mb path


C:\Users\jake\AppData\Local\Temp\ipykernel_7896\265206714.py:40: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  album_track_matches = pd.read_csv(ALBUM_TRACK_MATCH_TSV, sep='\t')


In [31]:

directory = 'D:\\Music\\MusicBee\\mb_playlists\\Year Top'
DRY_RUN = True
VERBOSE = True
SLEEP_TIME = 1



def validate_top_year_playlist(m3u_filepath):
  """Validates if a filepath is a valid top year playlist."""
  if not os.path.isfile(m3u_filepath):
    raise ValueError(f"File not found: {m3u_filepath}")

  if not m3u_filepath.endswith(".m3u"):
    raise ValueError(f"Invalid filename format. Must end with '.m3u': {m3u_filepath}")
  if ' top' not in m3u_filepath:
    raise ValueError(f"Invalid filename format. Must contain ' top': {m3u_filepath}")

  try:
    year_str = os.path.basename(m3u_filepath).split(" top")[0]
    if year_str.isdigit() and len(year_str) == 4:
      year = int(year_str)
    elif len(year_str) == 5 and year_str.endswith('s') and year_str[:-1].isdigit():
      year = year_str # Keep it as a string with "s"
    else:
      raise ValueError(f"Invalid year format in filename: {m3u_filepath}")
    return year

  except (ValueError, IndexError) as e:
    raise ValueError(f"Error extracting year from filename '{m3u_filepath}': {e}")




for filename in os.listdir(directory):
  filepath = os.path.join(directory, filename)
  if not filename.endswith('.m3u'): continue
  year = validate_top_year_playlist(filepath)

  if year != 2024: continue
  
  # Create or update year like playlist
  print(100*'*'+f"\nFound playlist for year: {year}")

  pl_name = f'y {year} top'
  m3u = pd.read_csv(filepath, index_col=None, sep='\t',  header=None)[0].unique().tolist()
  vids = []
  for path in m3u:
    if path in mb_path_map:
      fname = path.split('\\')[-3:]
      print(fname)
      vid = mb_path_map[path]
      vids.append(vid)
    
  print(f'[{year}] {len(vids)/len(m3u):.0%} found Musicbee m3u has {len(m3u)}',
        f'unique tracks, found {len(vids)} ytmusic vids')
  pl_id = Y.playlist_from_yt_vids(vids, pl_name=pl_name, sleep=3, public='PRIVATE',
                          desc=f'Top tracks for {year} (updated {DATE})',
                          dry=DRY_RUN, set_rating='LIKE', remove_dupes=True, verbose=VERBOSE)
  

****************************************************************************************************
Found playlist for year: 2024
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '01 - Skinshape - Stornoway.flac']
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '02 - Skinshape - Mulatu of Ethiopia.flac']
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '03 - Skinshape - Can You Play Me a Song-.flac']
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', "05 - Skinshape - It's About Time.flac"]
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '06 - Skinshape - How Can It Be-.flac']
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '07 - Skinshape - Ananda.flac']
['music_library', 'Skinshape - Another Side Of Skinshape (2024)', '09 - Skinshape - Massako.flac']
['music_library', 'Floating Points - Cascade (2024) [ZENDNL303] [WEB FLAC]', '01 - Floating Points - Vocoder (Club Mix).flac']
['musi

In [7]:
len(m3u)

271